# Star field focus check
Compare a sharp synthetic telescope frame with a slightly defocused copy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2024)
frame_height, frame_width = 96, 128
star_count = 40
sky_background_level = 12.0
read_noise_sigma = 1.5
star_rows = rng.uniform(4, frame_height - 4, star_count)
star_cols = rng.uniform(4, frame_width - 4, star_count)
star_brightness = rng.uniform(60, 240, star_count)
yy, xx = np.mgrid[0:frame_height, 0:frame_width]
print(f"Frame size: {frame_height} x {frame_width} pixels with {star_count} stars and background {sky_background_level}")
if star_count > 30:
    print("Dense field: overlapping halos are possible near bright stars.")
sharp_sigma = 0.9
soft_sigma = 2.2
print("Point spread widths (pixels):", sharp_sigma, "sharp vs", soft_sigma, "soft")

In [ ]:
sharp_frame = np.full((frame_height, frame_width), sky_background_level)
soft_frame = np.full((frame_height, frame_width), sky_background_level)
for row, col, brightness in zip(star_rows, star_cols, star_brightness):
    distance_sq = (yy - row) ** 2 + (xx - col) ** 2
    sharp_frame += brightness * np.exp(-distance_sq / (2 * sharp_sigma ** 2))
    soft_frame += (brightness * (sharp_sigma / soft_sigma) ** 2
                   * np.exp(-distance_sq / (2 * soft_sigma ** 2)))
sharp_frame += rng.normal(0, read_noise_sigma, sharp_frame.shape)
soft_frame += rng.normal(0, read_noise_sigma, soft_frame.shape)
sharp_frame = np.clip(sharp_frame, 0, 255)
soft_frame = np.clip(soft_frame, 0, 255)
print(f"Sharp frame range: {sharp_frame.min():.1f} to {sharp_frame.max():.1f}; mean {sharp_frame.mean():.2f}")
print(f"Soft frame range:  {soft_frame.min():.1f} to {soft_frame.max():.1f}; mean {soft_frame.mean():.2f}")
sharp_peak_fraction = (sharp_frame > 200).mean()
soft_peak_fraction = (soft_frame > 200).mean()
print(f"Pixels above 200: sharp {sharp_peak_fraction:.4f}, soft {soft_peak_fraction:.4f}")

## Focus metrics

In [ ]:
laplacian_kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=float)
padded_sharp = np.pad(sharp_frame, 1, mode="edge")
padded_soft = np.pad(soft_frame, 1, mode="edge")
laplacian_sharp = np.zeros_like(sharp_frame)
laplacian_soft = np.zeros_like(soft_frame)
for dy in range(3):
    for dx in range(3):
        weight = laplacian_kernel[dy, dx]
        if weight:
            laplacian_sharp += weight * padded_sharp[dy:dy + frame_height, dx:dx + frame_width]
            laplacian_soft += weight * padded_soft[dy:dy + frame_height, dx:dx + frame_width]
laplacian_variance_sharp = float(laplacian_sharp.var())
laplacian_variance_soft = float(laplacian_soft.var())
gradient_rows_sharp, gradient_cols_sharp = np.gradient(sharp_frame)
gradient_rows_soft, gradient_cols_soft = np.gradient(soft_frame)
gradient_energy_sharp = float(np.mean(gradient_rows_sharp ** 2 + gradient_cols_sharp ** 2))
gradient_energy_soft = float(np.mean(gradient_rows_soft ** 2 + gradient_cols_soft ** 2))
histogram_sharp, _ = np.histogram(sharp_frame, bins=64, range=(0, 255))
histogram_soft, _ = np.histogram(soft_frame, bins=64, range=(0, 255))
probability_sharp = histogram_sharp[histogram_sharp > 0] / histogram_sharp.sum()
probability_soft = histogram_soft[histogram_soft > 0] / histogram_soft.sum()
entropy_sharp = float(-(probability_sharp * np.log2(probability_sharp)).sum())
entropy_soft = float(-(probability_soft * np.log2(probability_soft)).sum())
print(f"Laplacian variance: sharp {laplacian_variance_sharp:.2f}, soft {laplacian_variance_soft:.2f}")
print(f"Gradient energy:    sharp {gradient_energy_sharp:.2f}, soft {gradient_energy_soft:.2f}")
print(f"Histogram entropy:  sharp {entropy_sharp:.3f} bits, soft {entropy_soft:.3f} bits")
if laplacian_variance_soft > 0:
    print(f"Laplacian ratio (sharp / soft): {laplacian_variance_sharp / laplacian_variance_soft:.2f}")

## Star shape measurements

In [ ]:
brightest_order = np.argsort(star_brightness)[::-1][:8]
half_window = 5
width_rows = []
for star_index in brightest_order:
    center_row = int(round(star_rows[star_index]))
    center_col = int(round(star_cols[star_index]))
    top = max(center_row - half_window, 0)
    bottom = min(center_row + half_window + 1, frame_height)
    left = max(center_col - half_window, 0)
    right = min(center_col + half_window + 1, frame_width)
    for label, frame in (("sharp", sharp_frame), ("soft", soft_frame)):
        patch = frame[top:bottom, left:right] - sky_background_level
        patch = np.clip(patch, 0, None)
        total_flux = patch.sum()
        if total_flux <= 0:
            continue
        patch_rows, patch_cols = np.mgrid[top:bottom, left:right]
        centroid_row = (patch * patch_rows).sum() / total_flux
        centroid_col = (patch * patch_cols).sum() / total_flux
        spread_sq = (patch * ((patch_rows - centroid_row) ** 2 + (patch_cols - centroid_col) ** 2)).sum() / total_flux
        fwhm_estimate = 2.3548 * np.sqrt(spread_sq / 2)
        width_rows.append((int(star_index), label, round(float(total_flux), 1), round(float(fwhm_estimate), 3)))
for star_index, label, flux, fwhm in width_rows:
    print(f"star {star_index:2d} {label:5s} flux {flux:9.1f} fwhm {fwhm:6.3f}")
sharp_widths = [row[3] for row in width_rows if row[1] == "sharp"]
soft_widths = [row[3] for row in width_rows if row[1] == "soft"]
print("Median FWHM sharp:", round(float(np.median(sharp_widths)), 3))
print("Median FWHM soft: ", round(float(np.median(soft_widths)), 3))

## Region by region comparison

In [ ]:
tile_rows, tile_cols = 3, 4
tile_height = frame_height // tile_rows
tile_width = frame_width // tile_cols
tile_report = []
for tile_row in range(tile_rows):
    for tile_col in range(tile_cols):
        row_slice = slice(tile_row * tile_height, (tile_row + 1) * tile_height)
        col_slice = slice(tile_col * tile_width, (tile_col + 1) * tile_width)
        sharp_tile = laplacian_sharp[row_slice, col_slice]
        soft_tile = laplacian_soft[row_slice, col_slice]
        sharp_score = float(sharp_tile.var())
        soft_score = float(soft_tile.var())
        ratio = sharp_score / soft_score if soft_score else float("inf")
        if ratio > 4:
            tile_verdict = "strongly softened"
        elif ratio > 2:
            tile_verdict = "softened"
        elif ratio > 1.2:
            tile_verdict = "slightly softened"
        else:
            tile_verdict = "similar"
        tile_report.append((tile_row, tile_col, round(sharp_score, 2), round(soft_score, 2), round(ratio, 2), tile_verdict))
for tile_row, tile_col, sharp_score, soft_score, ratio, tile_verdict in tile_report:
    print(f"tile ({tile_row},{tile_col}): sharp {sharp_score:9.2f} soft {soft_score:8.2f} ratio {ratio:6.2f} -> {tile_verdict}")
softened_tiles = sum(1 for row in tile_report if row[5] != "similar")
print(f"Tiles showing softening: {softened_tiles} of {len(tile_report)}")
row_profile_sharp = sharp_frame.mean(axis=1)
row_profile_soft = soft_frame.mean(axis=1)
profile_difference = float(np.abs(row_profile_sharp - row_profile_soft).mean())
print(f"Mean absolute row-profile difference: {profile_difference:.3f}")
column_profile_sharp = sharp_frame.mean(axis=0)
column_profile_soft = soft_frame.mean(axis=0)
column_difference = float(np.abs(column_profile_sharp - column_profile_soft).mean())
print(f"Mean absolute column-profile difference: {column_difference:.3f}")
correlation = float(np.corrcoef(sharp_frame.ravel(), soft_frame.ravel())[0, 1])
print(f"Pixel correlation between frames: {correlation:.4f}")
mean_sharp, mean_soft = sharp_frame.mean(), soft_frame.mean()
variance_sharp, variance_soft = sharp_frame.var(), soft_frame.var()
covariance = float(((sharp_frame - mean_sharp) * (soft_frame - mean_soft)).mean())
constant_one, constant_two = (0.01 * 255) ** 2, (0.03 * 255) ** 2
similarity_index = ((2 * mean_sharp * mean_soft + constant_one) * (2 * covariance + constant_two)) / ((mean_sharp ** 2 + mean_soft ** 2 + constant_one) * (variance_sharp + variance_soft + constant_two))
print(f"Global structural similarity: {similarity_index:.4f}")
contrast_sharp = float(sharp_frame.std() / sharp_frame.mean())
contrast_soft = float(soft_frame.std() / soft_frame.mean())
print(f"RMS contrast: sharp {contrast_sharp:.4f}, soft {contrast_soft:.4f}")
edge_threshold = np.percentile(np.abs(laplacian_sharp), 99)
edge_pixels_sharp = int((np.abs(laplacian_sharp) > edge_threshold).sum())
edge_pixels_soft = int((np.abs(laplacian_soft) > edge_threshold).sum())
print(f"Strong edge pixels above the sharp 99th percentile: sharp {edge_pixels_sharp}, soft {edge_pixels_soft}")

## Verdict rules

In [ ]:
metric_votes = []
laplacian_ratio = laplacian_variance_sharp / laplacian_variance_soft
if laplacian_ratio > 3:
    metric_votes.append(("laplacian", "defocused", laplacian_ratio))
elif laplacian_ratio > 1.5:
    metric_votes.append(("laplacian", "slightly defocused", laplacian_ratio))
else:
    metric_votes.append(("laplacian", "in focus", laplacian_ratio))
gradient_ratio = gradient_energy_sharp / gradient_energy_soft
if gradient_ratio > 3:
    metric_votes.append(("gradient", "defocused", gradient_ratio))
elif gradient_ratio > 1.5:
    metric_votes.append(("gradient", "slightly defocused", gradient_ratio))
else:
    metric_votes.append(("gradient", "in focus", gradient_ratio))
fwhm_ratio = float(np.median(soft_widths) / np.median(sharp_widths))
if fwhm_ratio > 2:
    metric_votes.append(("fwhm", "defocused", fwhm_ratio))
elif fwhm_ratio > 1.3:
    metric_votes.append(("fwhm", "slightly defocused", fwhm_ratio))
else:
    metric_votes.append(("fwhm", "in focus", fwhm_ratio))
if similarity_index < 0.6:
    metric_votes.append(("similarity", "defocused", similarity_index))
elif similarity_index < 0.85:
    metric_votes.append(("similarity", "slightly defocused", similarity_index))
else:
    metric_votes.append(("similarity", "in focus", similarity_index))
if edge_pixels_soft < edge_pixels_sharp * 0.25:
    metric_votes.append(("edges", "defocused", edge_pixels_soft))
elif edge_pixels_soft < edge_pixels_sharp * 0.6:
    metric_votes.append(("edges", "slightly defocused", edge_pixels_soft))
else:
    metric_votes.append(("edges", "in focus", edge_pixels_soft))
vote_counts = {}
for metric_name, vote, value in metric_votes:
    vote_counts[vote] = vote_counts.get(vote, 0) + 1
    print(f"{metric_name:10s} -> {vote:18s} ({value:.3f})")
if vote_counts.get("defocused", 0) >= 3:
    final_verdict = "The second frame is defocused; refocus before the next exposure."
elif vote_counts.get("defocused", 0) + vote_counts.get("slightly defocused", 0) >= 3:
    final_verdict = "The second frame is slightly soft; a small focus adjustment is advised."
else:
    final_verdict = "Both frames are acceptably sharp."
print("Vote counts:", dict(sorted(vote_counts.items())))
print("Verdict:", final_verdict)

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sharp_frame, cmap="gray")
axes[0].set_title("Sharp frame")
axes[1].imshow(soft_frame, cmap="gray")
axes[1].set_title("Soft frame")
axes[2].plot(row_profile_sharp, label="sharp")
axes[2].plot(row_profile_soft, label="soft")
axes[2].legend()
axes[2].set_title("Row profiles")
plt.tight_layout()
plt.show()
print("Figure drawn with", len(figure.axes), "panels")